In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime


In [2]:
# Load raw scraped dataset produced by the scraping pipeline
# This file contains unprocessed product data directly extracted from the website
df = pd.read_csv("../data/raw/whisky_raw.csv")
df.shape


(720, 13)

In [3]:
# Inspect dataset structure and data types
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   scraped_at      720 non-null    object 
 1   category        720 non-null    object 
 2   product_id      720 non-null    int64  
 3   name            720 non-null    object 
 4   brand           720 non-null    object 
 5   variant         720 non-null    object 
 6   price_gbp       720 non-null    float64
 7   unit_price_raw  698 non-null    object 
 8   region          720 non-null    object 
 9   promo_label     62 non-null     object 
 10  status          720 non-null    object 
 11  product_url     720 non-null    object 
 12  image_url       720 non-null    object 
dtypes: float64(1), int64(1), object(11)
memory usage: 73.3+ KB


In [4]:
# Preview first few rows to verify data loaded correctly
df.head(3)


,scraped_at,category,product_id,name,brand,variant,price_gbp,unit_price_raw,region,promo_label,status,product_url,image_url
0,2025-12-29 22:13:54,Single Malt,3121,Lagavulin 16 Year Old,Lagavulin,70cl / 43%,65.79,(£112.79 per litre),Islay,NaN,In Stock,https://www.thewhiskyexchange.com/p/3121/lagav...,https://img.thewhiskyexchange.com/380/lgvob.16...
1,2025-12-29 22:13:54,Single Malt,73311,Laphroaig 10 Year Old Cask Strength / Batch 016,Laphroaig,70cl / 58.5%,58.29,(£99.93 per litre),Islay,NaN,In Stock,https://www.thewhiskyexchange.com/p/73311/laph...,https://img.thewhiskyexchange.com/380/lrgob.10...
2,2025-12-29 22:13:54,Single Malt,56151,Talisker 2011 / 8 Year Old / Rum Finish / Spec...,Talisker,70cl / 57.9%,80.79,(£138.50 per litre),Island,NaN,In Stock,https://www.thewhiskyexchange.com/p/56151/tali...,https://img.thewhiskyexchange.com/380/talob.08...


## Data Cleaning 

This section standardizes raw scraped fields into analysis-ready formats. 

In [5]:
# Clean column names
df.columns = df.columns.str.strip()

df.columns


Index(['scraped_at', 'category', 'product_id', 'name', 'brand', 'variant',
       'price_gbp', 'unit_price_raw', 'region', 'promo_label', 'status',
       'product_url', 'image_url'],
      dtype='object')

In [6]:
# Drop the status column because it contains no variance (100% In Stock)
if df["status"].nunique() <= 1:
    df = df.drop(columns=["status"])
    print("Column 'status' dropped: No variance detected.")

Column 'status' dropped: No variance detected.


In [8]:
# Convert required columns to proper dtypes

df = df.assign(
    # Convert scraped_at to datetime
    scraped_at=pd.to_datetime(df["scraped_at"], errors="coerce"),

    # Convert product_id to integer
    product_id=pd.to_numeric(df["product_id"], errors="coerce").astype("Int64"),

    # Convert price_gbp to numeric (float)
    price_gbp=pd.to_numeric(df["price_gbp"], errors="coerce"),

    # Convert remaining columns to strings
    category=df["category"].astype("string"),
    name=df["name"].astype("string"),
    brand=df["brand"].astype("string"),
    variant=df["variant"].astype("string"),
    unit_price_raw=df["unit_price_raw"].astype("string"),
    region=df["region"].astype("string"),
    promo_label=df["promo_label"].astype("string"),
    product_url=df["product_url"].astype("string"),
    image_url=df["image_url"].astype("string"),
)

# Lightweight validation
print(df.dtypes)
print("\nMissing values after schema enforcement:")
print(df.isna().sum())



scraped_at        datetime64[ns]
category          string[python]
product_id                 Int64
name              string[python]
brand             string[python]
variant           string[python]
price_gbp                float64
unit_price_raw    string[python]
region            string[python]
promo_label       string[python]
product_url       string[python]
image_url         string[python]
dtype: object

Missing values after schema enforcement:
scraped_at          0
category            0
product_id          0
name                0
brand               0
variant             0
price_gbp           0
unit_price_raw     22
region              0
promo_label       658
product_url         0
image_url           0
dtype: int64


In [9]:
# Remove exact duplicate rows and verify product_id uniqueness

before = len(df)

# Define columns to consider for duplicate detection (exclude scraped_at)
dedup_cols = [c for c in df.columns if c != "scraped_at"]

# Drop duplicates based on all columns except scraped_at
df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)

after = len(df)

# Verify product_id uniqueness
is_unique = df["product_id"].is_unique
dup_count = df["product_id"].duplicated().sum()

print(f"Exact duplicate rows removed: {before - after}")
print(f"product_id unique after dedup: {is_unique}")
print(f"Remaining duplicate product_id count: {dup_count}")


Exact duplicate rows removed: 0
product_id unique after dedup: True
Remaining duplicate product_id count: 0


In [10]:
# Standardize missing values across all text columns

text_cols = df.select_dtypes(include="string").columns

# Strip whitespace first
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip())

# Standard missing tokens to normalize
MISSING_TOKENS = ["", "none", "n/a", "na", "nan", "null", "undefined"]

for col in text_cols:
    df[col] = df[col].where(
        ~df[col].str.lower().isin(MISSING_TOKENS),
        pd.NA
    )

# Validation
missing_after = df.isna().sum()
print(missing_after[missing_after > 0].sort_values(ascending=False))


promo_label       658
unit_price_raw     22
dtype: int64


In [11]:
# Handle inconsistencies with region

# Normalize region text early (strip + lowercase)
df["region"] = (
    df["region"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Blended products → region not applicable
df.loc[
    df["category"].str.lower() == "blended",
    "region"
] = pd.NA

# Valid regions for single malt
VALID_REGIONS = [
    "highland", "islay", "speyside", "lowland", "campbeltown", "island"
]

# Single Malt products with non-geographical region labels → set to NA
df.loc[
    (df["category"].str.lower() == "single malt")
    & (~df["region"].isin(VALID_REGIONS)),
    "region"
] = pd.NA

# Final formatting: Capitalize region names for presentation
df["region"] = df["region"].str.capitalize()

# Validation: Ensure only valid regions (and NA) remain
print("Unique regions after cleaning:")
print(df["region"].unique())

# Validation: Check counts to see how many rows were nullified
print("\nRegion value counts (including NAs):")
print(df["region"].value_counts(dropna=False))


Unique regions after cleaning:
<StringArray>
['Islay', 'Island', 'Campbeltown', 'Speyside', 'Highland', <NA>, 'Lowland']
Length: 7, dtype: string

Region value counts (including NAs):
region
<NA>           361
Speyside       123
Highland        97
Islay           77
Island          44
Lowland         10
Campbeltown      8
Name: count, dtype: Int64


In [12]:
# Brand Column Standardization 

# Normalize brand text (strip + collapse repeated whitespace)
df["brand"] = (
    df["brand"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Create a normalized helper key for case-insensitive mapping (logic-only)
brand_key_tmp = df["brand"].str.lower()

# Standardize known aliases / placeholders (case-insensitive)
brand_replacements_ci = {
    "chivas": "Chivas Regal",

    # placeholder / not-a-real-brand style values
    "various distilleries": "Independent/Unknown",
    "undisclosed distillery": "Independent/Unknown",
    "unknown distillery": "Independent/Unknown",
    "secret distillery": "Independent/Unknown",
}

df["brand"] = brand_key_tmp.map(brand_replacements_ci).fillna(df["brand"])

# Flag rows that should be excluded from "price by brand" charts
PLACEHOLDER_BRAND = "Independent/Unknown"
df["is_brand_placeholder"] = df["brand"].eq(PLACEHOLDER_BRAND)

# Validation 
print("\nPlaceholder brand count:")
print(df["is_brand_placeholder"].value_counts(dropna=False))

# Verify Chivas change 
print("\nChivas Regal count:")
print((df["brand"] == "Chivas Regal").sum())



Placeholder brand count:
is_brand_placeholder
False    717
True       3
Name: count, dtype: int64

Chivas Regal count:
32


In [13]:
# Parse bottle size (cl) and ABV (%) from variant

# Confirm which volume units appear in variant (handles 70cl, 70 cl, etc.)
units = df["variant"].str.extract(
    r"(\d+(?:\.\d+)?)\s*(ml|cl|l|litre|liter)",
    flags=re.IGNORECASE
)[1].str.lower()

print("Volume units found in 'variant':")
print(units.value_counts())

# Extract Volume (cl) 
df["bottle_size_cl"] = (
    df["variant"]
    .str.extract(r"(\d+(?:\.\d+)?)\s*cl", expand=False)
    .astype(float)
)

# Normalize to Litres (Standard Unit) 
df["bottle_size_l"] = df["bottle_size_cl"] / 100

# Extract ABV (%)
df["abv_percent"] = (
    df["variant"]
    .str.extract(r"(\d+(?:\.\d+)?)\s*%", expand=False)
    .astype(float)
)

#  Validation Checks 
print(f"Missing Volume (cl): {df['bottle_size_cl'].isna().sum()}")
print(f"Missing ABV (%):     {df['abv_percent'].isna().sum()}")

#  Preview results 
df[["variant", "bottle_size_cl", "bottle_size_l", "abv_percent"]].head()


Volume units found in 'variant':
1
cl    720
Name: count, dtype: Int64
Missing Volume (cl): 0
Missing ABV (%):     0


,variant,bottle_size_cl,bottle_size_l,abv_percent
0,70cl / 43%,70.0,0.7,43.0
1,70cl / 58.5%,70.0,0.7,58.5
2,70cl / 57.9%,70.0,0.7,57.9
3,70cl / 54.2%,70.0,0.7,54.2
4,70cl / 50.5%,70.0,0.7,50.5


In [14]:
# Parse unit price from unit_price_raw (standardize to GBP per litre)

# Extract numeric value from unit_price_raw
df["unit_price_gbp_per_litre"] = (
    df["unit_price_raw"]
      .astype("string")
      .str.replace("£", "", regex=False)
      .str.replace(",", "", regex=False)
      .str.extract(r"(\d+(?:\.\d+)?)", expand=False)
)
df["unit_price_gbp_per_litre"] = pd.to_numeric(df["unit_price_gbp_per_litre"], errors="coerce")

# Extract the reported basis after "per" or "/"
basis = (
    df["unit_price_raw"]
      .astype("string")
      .str.lower()
      .str.replace(",", "", regex=False)
      .str.extract(r"(?:per|/)\s*([0-9]+(?:\.[0-9]+)?\s*)?(ml|cl|l|litre|liter)", expand=False)
)

# basis[0] = quantity, basis[1] = unit
qty = pd.to_numeric(basis[0].astype("string").str.strip(), errors="coerce")
unit = basis[1].astype("string").replace({"liter": "litre"})

# If unit is litre/l and qty is missing → treat as 1 litre
mask_litre = ((unit == "litre") | (unit == "l")) & qty.isna()
qty.loc[mask_litre] = 1.0

# Convert all bases to "per litre" using scaling factors
# NOTE: Use np.nan (float missing), NOT pd.NA, because scale is float64
scale = pd.Series(np.nan, index=df.index, dtype="float64")

# per X ml  → value * (1000 / X)
mask_ml = (unit == "ml") & qty.notna()
scale.loc[mask_ml] = 1000 / qty.loc[mask_ml]

# per X cl  → value * (100 / X)
mask_cl = (unit == "cl") & qty.notna()
scale.loc[mask_cl] = 100 / qty.loc[mask_cl]

# per litre → value stays the same (or per 2 litres etc → divide by qty)
mask_l = ((unit == "litre") | (unit == "l")) & qty.notna()
scale.loc[mask_l] = 1.0 / qty.loc[mask_l]

# Apply scaling
df["unit_price_gbp_per_litre"] = (df["unit_price_gbp_per_litre"] * scale).round(2)

# Preview results
print(f"Rows processed: {len(df)} rows.")
print(f"Missing unit_price_gbp_per_litre: {df['unit_price_gbp_per_litre'].isna().sum()}")

df[["unit_price_raw", "unit_price_gbp_per_litre"]].head(10)



Rows processed: 720 rows.
Missing unit_price_gbp_per_litre: 22


,unit_price_raw,unit_price_gbp_per_litre
0,(£112.79 per litre),112.79
1,(£99.93 per litre),99.93
2,(£138.50 per litre),138.5
3,(£108.93 per litre),108.93
4,(£427.14 per litre),427.14
5,(£228.57 per litre),228.57
6,(£58.21 per litre),58.21
7,(£264.29 per litre),264.29
8,(£122.79 per litre),122.79
9,(£264.29 per litre),264.29


In [15]:
# Discount derivation from promo_label and reconstructing pre-discount prices

# Ensure price_gbp is numeric
df["price_gbp"] = pd.to_numeric(df["price_gbp"], errors="coerce")

# Extract explicit discount amounts (£) from promo_label (e.g., "£6 Off")
df["discount_amount_gbp"] = (
    df["promo_label"]
      .astype("string")
      .str.lower()
      .str.replace(",", "", regex=False)
      .str.extract(r"£\s*(\d+(?:\.\d+)?)", expand=False)
)
df["discount_amount_gbp"] = pd.to_numeric(df["discount_amount_gbp"], errors="coerce")

# Flag discounted products (is_discounted)
df["is_discounted"] = df["discount_amount_gbp"].notna()

# Reconstruct original (pre-discount) price
# If not discounted, discount_amount_gbp is NA → treated as 0
df["price_pre_discount_calc_gbp"] = df["price_gbp"] + df["discount_amount_gbp"].fillna(0)

# Quick validation
print("Discounted products:", int(df["is_discounted"].sum()))
print("Products without discounts (expected NA):", int(df["discount_amount_gbp"].isna().sum()))
print("Missing price_pre_discount_calc_gbp:", int(df["price_pre_discount_calc_gbp"].isna().sum()))

df[[
    "promo_label",
    "price_gbp",
    "discount_amount_gbp",
    "is_discounted",
    "price_pre_discount_calc_gbp"
]].head(10)


Discounted products: 55
Products without discounts (expected NA): 665
Missing price_pre_discount_calc_gbp: 0


,promo_label,price_gbp,discount_amount_gbp,is_discounted,price_pre_discount_calc_gbp
0,<NA>,65.79,<NA>,False,65.79
1,<NA>,58.29,<NA>,False,58.29
2,<NA>,80.79,<NA>,False,80.79
3,<NA>,63.54,<NA>,False,63.54
4,<NA>,249.17,<NA>,False,249.17
5,<NA>,133.33,<NA>,False,133.33
6,Special Offer: £6 Off!,38.96,6.0,True,44.96
7,<NA>,154.17,<NA>,False,154.17
8,<NA>,71.62,<NA>,False,71.62
9,<NA>,154.17,<NA>,False,154.17


## Price Integrity Validation: Unit Price Semantics

The site-reported unit price (`unit_price_raw`, parsed as `unit_price_gbp_per_litre`) presents a semantic ambiguity regarding its underlying valutaion logic. It is unclear whether this value is calculated from the current selling price, the pre-discount price, or whether it includes statutory tax adjustments (VAT) and retailer-specific normalization constants.

To ensure analytical correctness, we perform a validation analysis to identify the mathematical logic used by the retailer to compute unit prices.


In [16]:
# Unit Price Validation: Comparison with Net Price Models

TOL = 0.05  # Allowance for small rounding differences (£)

# Calculate Unit Price from Current Net Price
df["unit_net_current_calc"] = (df["price_gbp"] / df["bottle_size_l"]).round(2)

# Calculate Unit Price from Pre-Discount Net Price
df["unit_net_pre_calc"] = (df["price_pre_discount_calc_gbp"] / df["bottle_size_l"]).round(2)

# Check for matches against the site-reported unit price
matches_current = (abs(df["unit_net_current_calc"] - df["unit_price_gbp_per_litre"]) <= TOL).sum()
matches_pre = (abs(df["unit_net_pre_calc"] - df["unit_price_gbp_per_litre"]) <= TOL).sum()

print(f"Total products analyzed: {len(df)}")
print(f"Matches for Net Current Price model: {matches_current}")
print(f"Matches for Net Pre-Discount model: {matches_pre}")

if matches_current == 0 and matches_pre == 0:
    print("\n[OBSERVATION]: Site unit price does not match any net-price model.")
    print("This suggests the inclusion of VAT and other internal normalization constants.")

# Drop temporary validation columns (not used in downstream analysis)
df = df.drop(columns=[
    "unit_net_current_calc",
    "unit_net_pre_calc"
])

Total products analyzed: 720
Matches for Net Current Price model: 0
Matches for Net Pre-Discount model: 0

[OBSERVATION]: Site unit price does not match any net-price model.
This suggests the inclusion of VAT and other internal normalization constants.


### Conclusion: Unit Price Metric Integrity

The validation results confirm that the site-reported `unit_price_gbp_per_litre` does not align with a standard **Price ÷ Volume** calculation using either the current net selling price or the reconstructed pre-discount net price.

This mismatch indicates that the retailer’s unit price metric is calculated using a pricing basis that differs from the scraped net price fields. The site-reported value may incorporate statutory tax components (such as VAT), retailer-specific normalization rules, or other internal adjustments that are not explicitly disclosed in the available data.

Because the exact computational logic behind the site-reported unit price cannot be reliably inferred or reconstructed from the scraped dataset alone, this metric is considered semantically ambiguous and analytically unsafe for analytical use.


### Feature Engineering

After completing data cleaning, additional features are created to support analysis. These derived columns help standardize prices, identify discounted products, and group items by characteristics such as bottle size, alcohol strength, and price range. All features are calculated using existing, cleaned data without making assumptions beyond what is available. This ensures that the analysis is clear, accurate, and consistent.

In [17]:
# Price-derived features

# Original (pre-discount) price used as reference price
df["true_price_gbp"] = df["price_pre_discount_calc_gbp"]

# True unit price per litre (based on pre-discount price)
df["true_unit_price_per_l"] = (
    df["true_price_gbp"] / df["bottle_size_l"]
).round(2)

# Total alcohol units per bottle (UK definition: 10ml pure alcohol = 1 unit)
df["alcohol_units"] = df["bottle_size_l"] * df["abv_percent"] 

# Price per alcohol unit (based on true / pre-discount price)
df["true_price_per_alcohol_unit_gbp"] = (
    df["true_price_gbp"] / df["alcohol_units"]
).round(2)

# Price tier classification based on true (pre-discount) price
df["price_tier"] = pd.cut(
    df["true_price_gbp"],
    bins=[0, 40, 120, float("inf")],
    labels=["Budget", "Premium", "Luxury"],
    right=False
)

In [18]:
# Discount-related features

# Discount percentage based on true (pre-discount) price
df["discount_percent"] = (
    (df["true_price_gbp"] - df["price_gbp"]) / df["true_price_gbp"] * 100
).round(2)

# Set discount_percent to 0 for non-discounted products
df.loc[df["discount_amount_gbp"].isna(), "discount_percent"] = 0

# For discount band (Groups discounts into readable ranges)
df["discount_band"] = pd.cut(
    df["discount_percent"],
    bins=[-0.01, 0, 10, 20, 40, float("inf")],
    labels=["No Discount", "Low (≤10%)", "Medium (10–20%)", "High (20–40%)", "Very High (>40%)"]
)




In [19]:
# Age, volume and ABV related features

# Age statement extraction
df["age_years"] = (
    df["name"]
      .str.extract(r"\b(\d{1,2})\s*(?:year|years|yo)\b", flags=re.IGNORECASE, expand=False)
      .astype("float")
)

# Flag whether age is stated
df["is_age_stated"] = df["age_years"].notna()

# Age bands for age-stated products
df["age_band"] = pd.cut(
    df["age_years"],
    bins=[0, 10, 15, 18, 25, float("inf")],
    labels=["≤10", "11–15", "16–18", "19–25", "25+"]
)

# Bottle size bands (volume-based categorisation)
df["bottle_size_band"] = pd.cut(
    df["bottle_size_l"],
    bins=[0, 0.35, 0.7, 1.0, float("inf")],
    labels=["Small (≤35cl)", "Standard (70cl)", "Large (1L)", "Extra Large (>1L)"]
)

# ABV bands
df["abv_band"] = pd.cut(
    df["abv_percent"],
    bins=[0, 40, 43, 46, 50, float("inf")],
    labels=["≤40%", "40–43%", "43–46%", "46–50%", "50%+"]
)

# Cask strength indicator (derived from ABV threshold)
df["is_cask_strength"] = df["abv_percent"] >= 50





#### Saving the Cleaned Dataset

After completing data cleaning and feature engineering, the final processed dataset is saved as a cleaned CSV file. This file serves as the analysis-ready input for all subsequent exploratory analysis and visualizations

In [20]:
# Save cleaned dataframe to processed folder
output_path = "../data/processed/whisky_cleaned.csv"

df.to_csv(output_path, index=False)

print(f"File saved successfully to: {output_path}")




File saved successfully to: ../data/processed/whisky_cleaned.csv
